# Libraries integration
`!pip install azure-identity azure-ai-projects`

# Constants

In [1]:
# !az login

In [2]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./credentials.env")
model_name = os.environ["GPT4o_DEPLOYMENT_NAME"] #  the type must be "gpt-4o-2024-08-06": https://learn.microsoft.com/en-us/azure/ai-services/agents/how-to/tools/bing-grounding?tabs=python&pivots=overview#setup
project_connection_string = os.environ["PROJECT_CONNECTION_STRING"]

print(f'Project Connection String: <...{project_connection_string[-30:]}>')

Project Connection String: <...hub01-grp;mmai-swc-hub01-prj01>


# Retrieve credentials

In [3]:
from azure.identity import DefaultAzureCredential # requires pip install azure-identity
credential=DefaultAzureCredential()

# Connecto to the AI Foundry project

In [4]:
from azure.ai.projects import AIProjectClient
project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(), conn_str=project_connection_string
)
project_client.scope

{'subscription_id': 'eca2eddb-0f0c-4351-a634-52751499eeea',
 'resource_group_name': 'mmai-swc-hub01-grp',
 'project_name': 'mmai-swc-hub01-prj01'}

# Retrieve Bing connection
**NOTE**: the connection <`BING_CONNECTION_NAME`> has to be already associated to the project

In [5]:
from azure.ai.projects.models import BingGroundingTool
bing_connection = project_client.connections.get(connection_name=os.environ["BING_CONNECTION_NAME"])
bing = BingGroundingTool(connection_id=bing_connection.id)
print(f"bing.definitions: {bing.definitions}")

bing.definitions: [{'type': 'bing_grounding', 'bing_grounding': {'connections': [{'connection_id': '/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/mmai-swc-hub01-grp/providers/Microsoft.MachineLearningServices/workspaces/mmai-swc-hub01-prj01/connections/groundingwithbingsearchconnection'}]}}]


# Load or Create an AI Foundry Agent

In [6]:
# Agent creation
# Notices that FileSearchTool as tool and tool_resources must be added or the assistant unable to search the file

agent_id = "" # ex: asst_j1qWBdGsjbK4hHcO0M0n3M5p

if agent_id != "":
    agent = project_client.agents.get_agent(agent_id=agent_id)
else:
    agent = project_client.agents.create_agent(
        model=model_name,
        name="aiagent-PYTHON-bing",
        instructions="You are helpful assistant",
        tools=bing.definitions,
        headers={"x-ms-enable-preview": "true"}
    )

print(f"Agent: {agent}")

Agent: {'id': 'asst_O0fewUJDSe7NY6AOtDEU0Y2s', 'object': 'assistant', 'created_at': 1747117295, 'name': 'aiagent-PYTHON-bing', 'description': None, 'model': 'gpt-4o', 'instructions': 'You are helpful assistant', 'tools': [{'type': 'bing_grounding', 'bing_grounding': {'connections': [{'connection_id': '/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/mmai-swc-hub01-grp/providers/Microsoft.MachineLearningServices/workspaces/mmai-swc-hub01-prj01/connections/groundingwithbingsearchconnection'}]}}], 'top_p': 1.0, 'temperature': 1.0, 'tool_resources': {}, 'metadata': {}, 'response_format': 'auto'}


# Create the thread and attach a new message to it

In [7]:
# Create a thread
thread = project_client.agents.create_thread()
print(f"Created thread: {thread}\n")

Created thread: {'id': 'thread_PHG97ew73APyemvqplfyByYi', 'object': 'thread', 'created_at': 1747117295, 'metadata': {}, 'tool_resources': {}}



In [8]:
# Add a user message to the thread
message = project_client.agents.create_message(
    thread_id=thread.id, 
    role="user", 
    content="Quali sono i programmi TV stasera?", # "What is the top news today", "Quali sono i programmi TV stasera?"
)
print(f"Created message: {message}")

Created message: {'id': 'msg_CJRD75pzcLMf7mtFoYmj7Y8a', 'object': 'thread.message', 'created_at': 1747117295, 'assistant_id': None, 'thread_id': 'thread_PHG97ew73APyemvqplfyByYi', 'run_id': None, 'role': 'user', 'content': [{'type': 'text', 'text': {'value': 'Quali sono i programmi TV stasera?', 'annotations': []}}], 'attachments': [], 'metadata': {}}


# Run the agent syncrhonously

In [9]:
%%time
# Create and process assistant run in thread with tools
run = project_client.agents.create_and_process_run\
    (thread_id=thread.id, agent_id=agent.id, tool_choice='required',)

print(f"Run finished with status: {run.status}.\n\nRun: {run}")

if run.status == "failed":
    # Check if you got "Rate limit is exceeded.", then you want to get more quota
    print(f"Run failed: {run.last_error}")

Run finished with status: RunStatus.COMPLETED.

Run: {'id': 'run_ZZxRz9Nqlqxjp1G52fv2Be3j', 'object': 'thread.run', 'created_at': 1747117296, 'assistant_id': 'asst_O0fewUJDSe7NY6AOtDEU0Y2s', 'thread_id': 'thread_PHG97ew73APyemvqplfyByYi', 'status': 'completed', 'started_at': 1747117296, 'expires_at': None, 'cancelled_at': None, 'failed_at': None, 'completed_at': 1747117301, 'required_action': None, 'last_error': None, 'model': 'gpt-4o', 'instructions': 'You are helpful assistant', 'tools': [{'type': 'bing_grounding', 'bing_grounding': {'connections': [{'connection_id': '/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/mmai-swc-hub01-grp/providers/Microsoft.MachineLearningServices/workspaces/mmai-swc-hub01-prj01/connections/groundingwithbingsearchconnection'}]}}], 'tool_resources': {}, 'metadata': {}, 'temperature': 1.0, 'top_p': 1.0, 'max_completion_tokens': None, 'max_prompt_tokens': None, 'truncation_strategy': {'type': 'auto', 'last_messages': None}, 'incomplete_de

# Fetch messages from the thread after the agent run execution

In [10]:
from azure.ai.projects.models import MessageTextContent, MessageImageFileContent

if run.status == 'completed':    
    messages = project_client.agents.list_messages(thread_id=thread.id)
    messages_nr = len(messages.data)
    print(f"Here are the {messages_nr} messages:\n")
    
    for i, message in enumerate(reversed(messages.data), 1):
        j = 0
        print(f"\n===== MESSAGE {i} =====")
        for c in message.content:
            j +=1
            if (type(c) is MessageImageFileContent):
                print(f"\nCONTENT {j} (MessageImageFileContent) --> image_file id: {c.image_file.file_id}")
            elif (type(c) is MessageTextContent):
                print(f"\nCONTENT {j} (MessageTextContent) --> Text: {c.text.value}")
                for a in c.text.annotations:
                    print(f">>> Annotation in MessageTextContent {j} of message {i}: {a.text}\n")

else:
    print(f"Sorry, I can't proceed because the run status is {run.status}")

Here are the 2 messages:


===== MESSAGE 1 =====

CONTENT 1 (MessageTextContent) --> Text: Quali sono i programmi TV stasera?

===== MESSAGE 2 =====

CONTENT 1 (MessageTextContent) --> Text: I programmi TV di stasera includono:

- **Rai 1**: "Simon Coleman – Ultimo ballo" (21:25)
- **Rai 2**: Talk show tematico
- **Rai 3**: Documentari o approfondimenti culturali
- **Canali Mediaset**: Film o intrattenimento su Canale 5 e Italia 1.
Valuta opzioni specifiche tramite guide di palinsesto.


# Retrieve annotations from the messages

In [11]:
# Get the last message from the sender
last_msg = messages.get_last_message_by_role("assistant")
if last_msg:
    print(last_msg.content[0].text.value)

print(f"Number of annotation(s): {len(last_msg.content[0].text.annotations)}")

for annotation in last_msg.content[0].text.annotations:
    print(annotation["text"], annotation["url_citation"]["url"])

I programmi TV di stasera includono:

- **Rai 1**: "Simon Coleman – Ultimo ballo" (21:25)
- **Rai 2**: Talk show tematico
- **Rai 3**: Documentari o approfondimenti culturali
- **Canali Mediaset**: Film o intrattenimento su Canale 5 e Italia 1.
Valuta opzioni specifiche tramite guide di palinsesto.
Number of annotation(s): 0


# Teardown

In [12]:
# delete thread
project_client.agents.delete_thread(thread_id=thread.id)
print(f"{i} - Thread <{thread.id}> has been deleted")

# delete agent
project_client.agents.delete_agent(agent_id=agent.id)
print(f"{i} - Agent <{agent.id}> has been deleted")

2 - Thread <thread_PHG97ew73APyemvqplfyByYi> has been deleted
2 - Agent <asst_O0fewUJDSe7NY6AOtDEU0Y2s> has been deleted


# ALL-IN-ONE!

In [13]:
import os
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential # requires pip install azure-identity

def web_search(web_query:str,
        model_name:str=None,
        project_connection_string:str=None,
        credential:DefaultAzureCredential=DefaultAzureCredential(),
        connection_name:str=None,
        agent_id:str=None):
    
    load_dotenv("./credentials.env")
    model_name = os.environ["GPT4o_DEPLOYMENT_NAME"] if model_name is None else model_name
    project_connection_string = os.environ["PROJECT_CONNECTION_STRING"] if project_connection_string is None else project_connection_string
    connection_name = os.environ["BING_CONNECTION_NAME"] if connection_name is None else connection_name

    # Connecto to AI Foundry Project
    project_client = AIProjectClient.from_connection_string(
        credential=credential, conn_str=project_connection_string)
    
    # Retrieve Bing Connection
    bing_connection = project_client.connections.get(connection_name=os.environ["BING_CONNECTION_NAME"])
    bing = BingGroundingTool(connection_id=bing_connection.id)
    
    # Load or Create an AI Foundry Agent
    if agent_id is None:        
        agent = project_client.agents.create_agent(
            model=model_name,
            name="aiagent-PYTHON-bing",
            instructions='''
            You are a helpful agent that intelligently answers questions.
            ''',
            tools=bing.definitions,
            headers={"x-ms-enable-preview": "true"})    
    else:
        agent = project_client.agents.get_agent(agent_id=agent_id)        
        
    # Create the thread
    thread = project_client.agents.create_thread()
    
    # Attach a new message to the thread
    message = project_client.agents.create_message(
        thread_id=thread.id, 
        role="user", 
        content=web_query)    
    
    # Create and process assistant run in thread with tools
    run = project_client.agents.create_and_process_run\
        (thread_id=thread.id, agent_id=agent.id, tool_choice='required')
    
    # Get the last message from the sender
    messages = project_client.agents.list_messages(thread_id=thread.id)
    last_msg = messages.get_last_message_by_role("assistant")
    
    # Teardown: delete thread
    project_client.agents.delete_thread(thread_id=thread.id)
    print(f"{i} - Thread <{thread.id}> has been deleted")

    # Teardown: delete agent
    project_client.agents.delete_agent(agent_id=agent.id)

    return last_msg.content[0].text.value, last_msg.content[0].text.annotations


web_search("Qual è l'animale del 2023?")

2 - Thread <thread_10n0RiJC27KwQHBEir4rghAv> has been deleted


("L'animale dell'anno 2023 è il **moscardino** (*Muscardinus avellanarius*), un piccolo roditore scelto in Austria per sensibilizzare sulla protezione degli habitat naturali di questa specie【5:0†source】【5:8†source】.",
 [{'type': 'url_citation', 'text': '【5:0†source】', 'start_index': 188, 'end_index': 200, 'url_citation': {'url': 'https://www.naturaperte.com/questo-piccolissimo-roditore-e-stato-appena-votato-animale-dellanno-2023-in-austria/', 'title': 'Questo piccolissimo roditore è stato appena votato animale dell’anno ...'}},
  {'type': 'url_citation', 'text': '【5:8†source】', 'start_index': 200, 'end_index': 212, 'url_citation': {'url': 'https://www.greenme.it/animali/animali-selvatici/questo-piccolissimo-roditore-e-stato-appena-votato-animale-dellanno-2023-in-austria/', 'title': "Questo piccolissimo roditore è stato appena votato animale dell'anno ..."}}])